# Pipeline 1 (Thresholding)
Classifies every on-disk pixel of a single SDO observation into umbra, penumbra, plage, network or quiet Sun using intensity thresholds.

## 1. Imports:

In [ ]:
import os
import numpy as np
import pandas as pd
from astropy.time import Time

import matplotlib.pyplot as plt
import sunpy.map
from sunpy.net import Fido, attrs as a

from reproject import reproject_interp
from astropy import units as u

import cv2
from scipy import ndimage
from matplotlib.colors import ListedColormap, BoundaryNorm

import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
from skimage import measure

import time
stage_times = {}

## 2. Configuration:
Edit this cell as needed.

In [ ]:
JSOC_EMAIL = "your_email@example.com"   
DATA_DIR = './data'                        
os.makedirs(DATA_DIR, exist_ok=True)
TIME_START = '2014-01-01T00:00:00'                         
TIME_RANGE = a.Time(TIME_START, Time(TIME_START) + 1 * u.min)  

# Needed for download
MAX_RETRIES = 3            
MIN_FILE_SIZE = 1_000_000  # In bytes

# CLV correction
B_THRESHOLD = 50          
POLY_ORDER = 5            

# HMI segmentation (normalised Ic)
UMBRA_THRESHOLD = 0.50      # Below this = umbra
PENUMBRA_THRESHOLD = 0.85   # Below this = penumbra, above = quiet Sun
MIN_SPOT_SIZE = 5           # Min pixels per spot region

# AIA segmentation (QS median + n sigma)
NETWORK_SIGMA = 1.0         # Network threshold
PLAGE_SIGMA = 2.0           # Plage threshold
MIN_NETWORK_SIZE = 4        # Min pixels per network region
MIN_PLAGE_SIZE = 450        # Min pixels per plage region

# Display of the disks after segmentation:
DISPLAY_STEP = 2      
DISPLAY_SIZE = 800    
SEG_CLASSES = {       
    'off-disk': (-1, '#dddddd'),
    'QS':       (0,  '#90e0ef'),
    'network':  (1,  '#80b918'),
    'plage':    (2,  '#c77dff'),
    'penumbra': (3,  '#FF9900'),
    'umbra':    (4,  '#FF4444'),
}

# Contour width of the overlay on CLV corrected disks
CONTOUR_WIDTH = 1.2     

## 3. Functions:

In [ ]:
# Function for downloading the FITs files
def search_and_fetch(query, name, max_retries=MAX_RETRIES):
    for attempt in range(1, max_retries + 1):
        print(f"\n{name}: attempt {attempt}/{max_retries}...")
        result = Fido.search(query)
        n_results = len(result[0]) if len(result) else 0
        print(f"{name}: {n_results} results found")

        if n_results == 0:
            print(f"{name}: no results, retrying search...")
            time.sleep(5)
            continue

        files = Fido.fetch(result, path=f'{DATA_DIR}/{{file}}', overwrite=True)

        good_files = []
        for f in files:
            size = os.path.getsize(f)
            if size > MIN_FILE_SIZE:
                good_files.append(f)
            else:
                print(f"  BAD (size={size} bytes): {f} -- removing")
                os.remove(f)

        if good_files:
            print(f"{name}: SUCCESS -- {len(good_files)} valid file(s)")
            return good_files

        print(f"{name}: no valid files, retrying...")
        time.sleep(5)

    print(f"{name}: FAILED after {max_retries} attempts")
    return []

# Function for applying CLV correction
def clv_correct(target_map, target_data, mag_map, b_threshold, poly_order, mag_data_native=None):
    ny, nx = target_data.shape
    y, x = np.mgrid[0:ny, 0:nx]
    x0 = target_map.meta['crpix1'] - 1
    y0 = target_map.meta['crpix2'] - 1
    r_sun_px = target_map.meta['rsun_obs'] / target_map.meta['cdelt1']
    r = np.sqrt((x - x0)**2 + (y - y0)**2)
    mu = np.sqrt(np.clip(1 - (r / r_sun_px)**2, 0, 1))
    on_disk = r <= r_sun_px

    if mag_data_native is not None and mag_data_native.shape == target_data.shape:
        mag_reproj = mag_data_native
    else:
        mag_reproj, _ = reproject_interp(mag_map, target_map.wcs, shape_out=target_data.shape)

    quiet_sun_mask = (np.abs(mag_reproj) < b_threshold) & on_disk & np.isfinite(target_data)
    coeffs = np.polyfit(mu[quiet_sun_mask], target_data[quiet_sun_mask], poly_order)
    ctl_profile = np.poly1d(coeffs)(mu)
    ctl_profile[~on_disk] = np.nan
    corrected = target_data / ctl_profile
    return corrected, ctl_profile, quiet_sun_mask

# Function to remove connected regions smaller than min_size pixels.
def _filter_by_size(mask, min_size):
    if not np.any(mask):
        return mask
    lbl, n = ndimage.label(mask)
    if n == 0:
        return mask
    sizes = ndimage.sum(mask, lbl, range(1, n + 1))
    valid_labels = np.where(sizes >= min_size)[0] + 1
    return np.isin(lbl, valid_labels)

# Segment the features in from the HMI IC data
def segment_hmi(data, on_disk, umbra_thresh, penumbra_thresh, min_size):
    blurred = cv2.GaussianBlur(data.astype(np.float32), (7, 7), 0)

    umbra_raw = (blurred < umbra_thresh) & on_disk
    umbra = _filter_by_size(umbra_raw, min_size)

    dark_region_raw = (blurred < penumbra_thresh) & on_disk
    dark_region = _filter_by_size(dark_region_raw, min_size)
    spot = ndimage.binary_fill_holes(dark_region)
    penumbra = spot & (~umbra)

    return {"umbra": umbra, "penumbra": penumbra, "spot": spot}

# Segment the features from the AIA 1700 data
def segment_aia(data, spot_mask, on_disk, network_sigma, plage_sigma, min_plage_size, min_network_size):
    valid_qs_region = on_disk & (~spot_mask) & np.isfinite(data)
    qs_median = np.nanmedian(data[valid_qs_region])
    qs_std = np.nanstd(data[valid_qs_region])

    plage_thresh = qs_median + plage_sigma * qs_std
    network_thresh = qs_median + network_sigma * qs_std

    aia_nodark = np.where(spot_mask, np.nan, data.astype(float))
    blurred = cv2.GaussianBlur(np.nan_to_num(aia_nodark).astype(np.float32), (3, 3), 0)

    plage_raw = (blurred > plage_thresh) & on_disk & (~spot_mask)
    plage = _filter_by_size(plage_raw, min_plage_size)

    network_raw = (blurred > network_thresh) & (~plage) & on_disk & (~spot_mask)
    network = _filter_by_size(network_raw, min_network_size)

    qs = on_disk & (~spot_mask) & (~plage) & (~network) & np.isfinite(data)

    return {"plage": plage, "network": network, "qs": qs}

# Scales an image to 0-1 between vmin and vmax and colours it with a colormap
def _to_rgb(img, cmap, vmin, vmax):
    norm = np.clip((np.nan_to_num(img) - vmin) / (vmax - vmin), 0, 1)
    rgb = (plt.get_cmap(cmap)(norm)[..., :3] * 255).astype(np.uint8)
    rgb[~np.isfinite(img)] = 0
    return rgb

# Colours each pixel of the label map by its class colour in SEG_CLASSES
def _seg_to_rgb(seg_map):
    rgb = np.zeros(seg_map.shape + (3,), dtype=np.uint8)
    for val, col in SEG_CLASSES.values():
        rgb[seg_map == val] = [int(col[i:i+2], 16) for i in (1, 3, 5)]
    return rgb

# Makes a plotly figure from a downsampled, compressed image, keeping full-resolution pixel coordinates
def _image_fig(rgb, title, step, size, fmt='png'):
    H, W = rgb.shape[:2]
    fig = px.imshow(rgb[::step, ::step], binary_string=True, binary_format=fmt, origin='lower',
                    x=np.arange(0, W, step), y=np.arange(0, H, step))
    fig.update_layout(title=title, width=size, height=size,
                      margin=dict(l=10, r=10, t=40, b=10))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return fig

# Draws the outline of a mask
def _add_contour(fig, mask, name, visible=True):
    col = SEG_CLASSES[name][1]
    contours = [measure.approximate_polygon(c, tolerance=0.5)
                for c in measure.find_contours(mask.astype(np.uint8), 0.5)]
    if not contours:
        return
    pts = np.concatenate([np.vstack([c, [[np.nan, np.nan]]]) for c in contours])
    fig.add_trace(go.Scatter(x=pts[:, 1], y=pts[:, 0], mode='lines', name=name,
                             line=dict(color=col, width=CONTOUR_WIDTH), hoverinfo='skip',
                             visible=True if visible else 'legendonly'))

# Builds tabs of the results
def segmentation_tabs(seg_map, ic_norm, aia_data, masks_hmi, masks_aia, step, size):
    fig_seg = _image_fig(_seg_to_rgb(seg_map), 'Segmentation', step, size)
    for name, (_, col) in SEG_CLASSES.items():
        fig_seg.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name=name,
                                     marker=dict(symbol='square', size=12, color=col)))

    aia_vmin, aia_vmax = np.nanpercentile(aia_data, [1, 99])
    aia_rgb = _to_rgb(aia_data, 'sdoaia1700', aia_vmin, aia_vmax)
    net = masks_aia['network']
    net_col = np.array([int(SEG_CLASSES['network'][1][i:i+2], 16) for i in (1, 3, 5)])
    aia_rgb[net] = (0.4 * aia_rgb[net] + 0.6 * net_col).astype(np.uint8)
    fig_aia = _image_fig(aia_rgb, 'CLV-corrected AIA 1700', step, size, fmt='jpg')
    fig_aia.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name='network (shaded)',
                                 marker=dict(symbol='square', size=12, color=SEG_CLASSES['network'][1])))
    _add_contour(fig_aia, masks_aia['plage'], 'plage')

    fig_hmi = _image_fig(_to_rgb(ic_norm, 'gray', 0.3, 1.3), 'CLV-corrected HMI Ic',
                         step, size, fmt='jpg')
    _add_contour(fig_hmi, masks_hmi['umbra'], 'umbra')
    _add_contour(fig_hmi, masks_hmi['penumbra'], 'penumbra')


    figs = {'Segmentation': fig_seg, 'AIA 1700': fig_aia, 'HMI Ic': fig_hmi}
    tab = widgets.Tab()
    outputs = []
    for fig in figs.values():
        out = widgets.Output()
        with out:
            fig.show()
        outputs.append(out)
    tab.children = outputs
    for i, title in enumerate(figs):
        tab.set_title(i, title)
    return tab

## 4. Download:

In [ ]:
t0 = time.time()

query_aia = TIME_RANGE & a.jsoc.Series('aia.lev1_uv_24s') & a.jsoc.Wavelength(1700*u.angstrom) & a.jsoc.Notify(JSOC_EMAIL)
query_ic  = TIME_RANGE & a.jsoc.Series('hmi.Ic_720s') & a.jsoc.Notify(JSOC_EMAIL)
query_mag = TIME_RANGE & a.jsoc.Series('hmi.M_720s') & a.jsoc.Notify(JSOC_EMAIL)

files_aia = search_and_fetch(query_aia, "AIA 1700")
files_ic  = search_and_fetch(query_ic, "HMI Ic")
files_mag = search_and_fetch(query_mag, "HMI Mag")

if not (files_aia and files_ic and files_mag):
    raise RuntimeError("One or more downloads failed -- check summary above before continuing")
stage_times['Download'] = time.time() - t0

## 5. Loading raw disk images as sunpy maps:

In [ ]:
t0 = time.time()
aia_map = sunpy.map.Map(files_aia[0])
ic_map  = sunpy.map.Map(files_ic[0])
mag_map = sunpy.map.Map(files_mag[0])

aia_data = aia_map.data.astype(float)
ic_data  = ic_map.data.astype(float)
mag_data = mag_map.data.astype(float)
stage_times['Making sunpy maps'] = time.time() - t0


## 6. Limb darkening removal:

In [ ]:
t0 = time.time()

# CLV correction (Ic uses native magnetogram; AIA reprojects it inside clv_correct)
ic_corrected, ic_ctl, ic_mask = clv_correct(ic_map, ic_data, mag_map,
                                            b_threshold=B_THRESHOLD, 
                                            poly_order=POLY_ORDER,
                                            mag_data_native=mag_data)

aia_corrected, aia_ctl, aia_mask = clv_correct(aia_map, aia_data, mag_map,
                                               b_threshold=B_THRESHOLD, 
                                               poly_order=POLY_ORDER)

# Plot comparisons
fig, axes = plt.subplots(2, 2, figsize=(14, 14))

axes[0,0].imshow(ic_data, origin='lower', cmap='gray')
axes[0,0].set_title('Raw HMI Ic')

axes[0,1].imshow(ic_corrected, origin='lower', cmap='gray', vmin=0.5, vmax=1.5)
axes[0,1].set_title(f'CLV-corrected Ic (B={B_THRESHOLD}G)')

aia_vmin, aia_vmax = np.nanpercentile(aia_data, [1, 99])
axes[1,0].imshow(aia_data, origin='lower', cmap='sdoaia1700', vmin=aia_vmin, vmax=aia_vmax)
axes[1,0].set_title('Raw AIA 1700')

axes[1,1].imshow(aia_corrected, origin='lower', cmap='sdoaia1700', vmin=0.5, vmax=1.5)
axes[1,1].set_title(f'CLV-corrected AIA 1700 (B={B_THRESHOLD}G)')

plt.tight_layout()
plt.savefig('clv_comparison_both_channels.png', dpi=600, bbox_inches='tight')
plt.show()
stage_times['CLV correction'] = time.time() - t0

## 7. Rotation & reprojection:

In [ ]:
t0 = time.time()

# Rotate CLV-corrected AIA to solar north (order=3)
aia_corrected_map = sunpy.map.Map(aia_corrected, aia_map.meta)
aia_rotated = aia_corrected_map.rotate(order=3)
print("Rotated AIA shape:", aia_rotated.data.shape)

# Reproject CLV-corrected Ic and magnetogram onto the rotated AIA grid
ic_corrected_map = sunpy.map.Map(ic_corrected, ic_map.meta)
ic_reprojected, _  = reproject_interp(ic_corrected_map, aia_rotated.wcs, shape_out=aia_rotated.data.shape)
mag_reprojected, _ = reproject_interp(mag_map, aia_rotated.wcs, shape_out=aia_rotated.data.shape)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(aia_rotated.data, origin='lower', cmap='sdoaia1700', vmin=0.5, vmax=1.5)
axes[0].set_title('Rotated CLV-corrected AIA 1700')

axes[1].imshow(ic_reprojected, origin='lower', cmap='gray')
axes[1].set_title('HMI Ic reprojected onto AIA grid')

axes[2].imshow(mag_reprojected, origin='lower', cmap='gray', vmin=-500, vmax=500)
axes[2].set_title('Magnetogram reprojected onto AIA grid')

plt.tight_layout()
plt.show()
stage_times['Rotate + reproject'] = time.time() - t0

## 8. Segmentation:

In [ ]:
t0 = time.time()

# Normalise Ic to full-disk median
on_disk_full = np.isfinite(ic_reprojected)
ic_norm = ic_reprojected / np.nanmedian(ic_reprojected[on_disk_full])

# Masks
masks_hmi = segment_hmi(ic_norm, on_disk_full,
                        umbra_thresh=UMBRA_THRESHOLD, penumbra_thresh=PENUMBRA_THRESHOLD,
                        min_size=MIN_SPOT_SIZE)
masks_aia = segment_aia(aia_rotated.data, masks_hmi['spot'], on_disk_full,
                        network_sigma=NETWORK_SIGMA, plage_sigma=PLAGE_SIGMA,
                        min_plage_size=MIN_PLAGE_SIZE, min_network_size=MIN_NETWORK_SIZE)

# Pixel counts per class
pixel_counts = pd.DataFrame(
    [(name, val, (seg_map == val).sum()) for name, (val, _) in SEG_CLASSES.items() if val >= 0],
    columns=['Class', 'Label', 'Pixels']).set_index('Class')
pixel_counts['% of disk'] = 100 * pixel_counts['Pixels'] / pixel_counts['Pixels'].sum()
pixel_counts.loc['TOTAL'] = ['-', pixel_counts['Pixels'].sum(), 100.0]
display(pixel_counts.round(2))

# Label map: -1 off-disk, 0 QS, 1 network, 2 plage, 3 penumbra, 4 umbra
seg_map = np.full(ic_norm.shape, -1)
seg_map[masks_aia['qs']] = 0
seg_map[masks_aia['network']] = 1
seg_map[masks_aia['plage']] = 2
seg_map[masks_hmi['penumbra']] = 3
seg_map[masks_hmi['umbra']] = 4

stage_times['Segmentation'] = time.time() - t0

## 9. Plotting segmentation:

In [ ]:
t0 = time.time()
tabs = segmentation_tabs(seg_map, ic_norm, aia_rotated.data, masks_hmi, masks_aia, step=DISPLAY_STEP, size=DISPLAY_SIZE)
display(tabs)
stage_times['Plotting'] = time.time() - t0

## 10. Timer summary table:

In [ ]:
timings = pd.DataFrame({'Time (s)': stage_times}).rename_axis('Stage')
timings['% of total'] = 100 * timings['Time (s)'] / timings['Time (s)'].sum()
timings.loc['TOTAL'] = [timings['Time (s)'].sum(), 100.0]
display(timings.round(2))